
# 📦 Notebook 1 — Data Generation & Understanding

**Mục tiêu notebook:**
- Load `pred_repurchase_dataset.csv` (đã xử lý anti-leakage) và kiểm tra schema
- Hiểu cấu trúc 20 cột và ý nghĩa business của từng cột
- Mô tả bài toán dự đoán tái mua hàng (binary classification)
- Giải thích nguồn gốc nhãn `repurchase` (Otsu thresholding T*=4.02 trên purchase_count)
- Lưu bản copy vào `data/synthetic_data.csv` để EDA ở Notebook 2

---

## 🗺️ Lộ trình 15 bước — Pipeline ML đầy đủ

| # | Bước | Notebook |
|:-:|------|----------|
| **1** | **Đặt vấn đề** | **📌 NB1** |
| **2** | **Thu thập dữ liệu** | **📌 NB1** |
| 3 | Tiền xử lý & EDA | NB2 |
| 4 | Định nghĩa Target | NB2 |
| 5 | Feature Engineering | NB3 |
| 6 | Feature Selection | NB3 |
| 7 | Train/Test Split + Scale | NB3 |
| 8 | Chọn mô hình | NB3 |
| 9 | Huấn luyện | NB3 |
| 10 | Đánh giá baseline | NB3 |
| 11 | Feature Reduction | NB3 |
| 12 | So sánh & Phân tích sâu | NB4 |
| 13 | Kết luận Model tốt nhất | NB4 |
| 14 | Threshold Optimization | NB4 |
| 15 | KPI Check | NB4 |

> 💡 **Notebook này bao phủ: Bước 1 & 2 / 15**

---
## 🎨 Visual Identity — Quy ước màu sắc toàn dự án

> Áp dụng nhất quán trên **tất cả 4 notebooks** để đảm bảo portfolio cohesion.

### Màu chính — Class colors

| Token | Hex | Dùng cho | Ví dụ |
|-------|-----|----------|-------|
| `c0` | `#F7CAD0` | Fill — Class 0 (Không tái mua) | Boxplot, histogram, bar |
| `c0_edge` | `#C97680` | Viền — Class 0 | Edge/outline |
| `c1` | `#A0C4FF` | Fill — Class 1 (Tái mua) | Boxplot, histogram, bar |
| `c1_edge` | `#4A80C4` | Viền — Class 1 | Edge/outline |

### Màu nền & layout

| Token | Hex | Dùng cho |
|-------|-----|----------|
| `bg` | `#FAFAFA` | Figure & axes background |
| `grid` | `#EAEAEA` | Gridlines (rất nhẹ) |
| `text` | `#3A3A3A` | Main text, axis labels, regular titles |
| `sub` | `#888888` | Caption, annotation, secondary info |

### Màu nhấn — Semantic highlight

| Token | Hex | Ý nghĩa |
|-------|-----|---------|
| `hi_title` | `#D96C6C` | **Title đỏ** — feature có class separation / predictive signal mạnh |
| `hi_bdr` | `#B0CCEE` | Border nhẹ — subplot chứa feature quan trọng |

### Quy tắc sử dụng

```
📌 RULE 1 — Class colors:
   Class 0 (không tái mua) → LUÔN dùng #F7CAD0 (pastel pink)
   Class 1 (tái mua)       → LUÔN dùng #A0C4FF (pastel sky blue)

📌 RULE 2 — Highlight title:
   Feature có class separation mạnh → title màu #D96C6C (bold)
   Feature thông thường             → title màu #3A3A3A (regular)
   KHÔNG dùng ⭐ hay ký hiệu đặc biệt trong title

📌 RULE 3 — Background:
   Figure facecolor = #FAFAFA   |   Grid color = #EAEAEA
   Không dùng white (#FFFFFF) hay transparent

📌 RULE 4 — Consistency:
   Mọi chart trong dự án đều dùng chung dict VI = {...}
   Khai báo một lần ở đầu cell, tham chiếu qua key
```

### Áp dụng theo notebook

| Notebook | Charts | Colors dùng |
|----------|--------|-------------|
| **NB1** (Data Generation) | Schema, label distribution | `c0`, `c1`, `bg`, `text` |
| **NB2** (EDA) | Boxplot grid, heatmap, distribution | All tokens + `hi_title`, `hi_bdr` |
| **NB3** (Model Building) | Feature importance, learning curves | `c0`, `c1`, `bg`, `text`, `sub` |
| **NB4** (Evaluation) | ROC, PR, Confusion Matrix, KPI | `c0`, `c1`, `bg`, `text`, `sub` |



---
## 📌 Bước 1 / 15 — Đặt vấn đề & Mô tả Bài toán Business

### ❓ Câu hỏi kinh doanh cốt lõi

> **"Dựa trên hành vi tương tác của khách hàng với một category sản phẩm,**  
> **khách có quay lại mua thêm sản phẩm trong chính category đó không?"**

Một nền tảng e-commerce Việt Nam muốn **cá nhân hóa chiến lược giữ chân khách hàng** theo từng danh mục sản phẩm. Thay vì chạy khuyến mãi đại trà, họ cần biết: *"Khách hàng X có khả năng mua lại trong danh mục Y không?"*

---

### 🏢 Bối cảnh & Business Impact

- **Vấn đề:** ~67% khách không quay lại category sau lần mua đầu → chi phí acquisition cao bị lãng phí
- **Giải pháp:** Dự đoán sớm khách nào có khả năng tái mua category → gửi voucher/reminder đúng người
- **ROI:** Gửi voucher đại trà tốn 30,000 VNĐ/người × toàn bộ khách; nếu target đúng → chỉ gửi ~33% → tiết kiệm ~67% chi phí CRM

| Dự đoán | Thực tế | Hành động | Hệ quả |
|---------|---------|-----------|--------|
| 1 (tái mua) | 1 ✅ | Gửi voucher loyalty → giữ chân | Hiệu quả cao |
| 1 (tái mua) | 0 ❌ | Voucher gửi nhầm | Chi phí marketing lãng phí |
| 0 (không tái mua) | 1 ❌ | Không gửi gì | Bỏ lỡ cơ hội upsell → mất doanh thu |

---

### 🔍 Hiểu đúng "Tái mua" trong bài toán này

| Khái niệm | Định nghĩa trong bài toán này |
|-----------|-------------------------------|
| **Tái mua (repurchase = 1)** | Khách đã mua ở một category → **quay lại mua thêm ≥ 5 lần** trong cùng category đó |
| **Không tái mua (repurchase = 0)** | Khách chỉ mua 1–4 lần rồi dừng, hoặc không quay lại category đó nữa |
| **Đơn vị phân tích** | Mỗi cặp **(user, category)** — không phải toàn bộ lịch sử mua hàng |
| **Tín hiệu dự đoán** | Các feature trong data (KHÔNG dùng purchase_count) |

> 🎯 **Tại sao cùng category?** Khách mua áo thun → có tái mua áo thun nữa không?  
> Đây là **category loyalty** — khác với cross-sell (mua thêm category khác).  
> Model này giúp team CRM biết nên **re-target** khách vào đúng category họ hay mua.

---

### 🤖 Định nghĩa bài toán ML

| Hạng mục | Chi tiết |
|----------|---------|
| **Task** | Binary Classification |
| **Input (X)** | 16 raw behavioral features + 15 derived features (xem Bước 5) |
| **Output (y)** | `repurchase`: 0 = không tái mua category, 1 = tái mua category |
| **Nhãn nguồn gốc** | Otsu thresholding trên `purchase_count` → **T\* = 4.02** → label=1 nếu ≥ 5 lần |
| **Đơn vị** | 1 dòng = 1 user × 1 category (aggregate của toàn bộ hành vi trong category đó) |

---

### 📊 KPI mục tiêu

Vì class imbalance (~33/67), **F1-Score** là metric chính — cân bằng Precision và Recall:
- **Precision** cao → ít lãng phí voucher (ít FP)
- **Recall** cao → ít bỏ lỡ khách tái mua tiềm năng (ít FN)

| Metric | Vai trò ở giai đoạn NB1 | Cơ sở phân tích hiện có |
|--------|--------------------------|------------------------|
| F1-Score | **Primary metric** | Dataset có class imbalance khoảng **67/33**; nếu luôn dự đoán lớp 0 thì Accuracy vẫn cao nhưng mô hình không có giá trị thực tiễn |
| Recall | **Guardrail metric** | Cần theo dõi riêng nguy cơ bỏ sót khách có khả năng tái mua; mức chi phí FN/FP cụ thể sẽ được lượng hóa ở notebook đánh giá |
| ROC-AUC | **Ranking metric** | Đo khả năng phân biệt hai nhóm khách hàng một cách độc lập với threshold, phù hợp để so sánh các mô hình ở giai đoạn sau |
##
> 📌 **Lưu ý phương pháp:** Ở notebook đầu tiên, ta mới chỉ có **business brief + mô tả dữ liệu + phân phối target sơ bộ**, nên **chưa đủ cơ sở khoa học để chốt trước các ngưỡng KPI định lượng** như F1 ≥ x hay ROC-AUC ≥ y.
>
> Các **ngưỡng KPI số** sẽ được xác lập sau khi hoàn thành:
> 1. **EDA và kiểm tra chất lượng dữ liệu** (NB2)
> 2. **Baseline model và feature engineering** (NB3)
> 3. **Model evaluation + threshold tuning + business cost analysis** (NB4)
>
> Vì vậy, ở **NB1** phần này nên được hiểu là **tiêu chí đánh giá ban đầu** chứ không phải **KPI kết quả đã chốt sẵn**.

**3 Models:** Logistic Regression (baseline) → XGBoost → LightGBM


---
## 📌 Bước 2 / 15 — Thu thập dữ liệu (Data Collection)

**Nguồn dữ liệu:** `pred_repurchase_dataset.csv` — E-commerce behavioral dataset

### Cấu trúc dữ liệu

> **Mỗi dòng = 1 user × 1 category** — aggregate toàn bộ hành vi của user trong category đó.

| Nhóm cột | Ví dụ | Vai trò |
|----------|-------|---------|
| **Behavioral** | `total_view`, `total_click`, `total_cart`, `total_wishlist` | Tín hiệu hành vi trong category |
| **Purchase aggregate** | `avg_purchase_value`, `avg_price`, `avg_discount` | Thống kê giá/mua trong category |
| **Category profile** | `category_share`, `user_total_categories`, `unique_brands` | Mức độ tập trung vào category |
| **User profile** | `age`, `gender`, `membership_tier`, `region` | Đặc điểm demographics |
| **Rating** | `avg_rating` | Hài lòng trung bình với sản phẩm trong category (~28% missing) |
| **Target** | `repurchase` | 0/1 — tái mua trong category không? |

**Quy mô:** 61,728 cặp (user, category) × 20 cột  
**Class distribution:** 67.2% không tái mua (label=0) · 32.8% tái mua (label=1)  
**Otsu T\* = 4.02** → `purchase_count ≥ 5` → label=1

**Anti-leakage đã xử lý:**
- `purchase_count` → **ĐÃ XÓA** (dùng để tạo nhãn — nếu giữ = biết đáp án trước)
- `total_interactions` → **ĐÃ XÓA** (= view + click + cart + wishlist + purchase_count → chứa purchase_count)

### 1. Load dữ liệu nguồn


In [1]:
# ============================================================
# SECTION 1: Load dữ liệu
# ============================================================
df = pd.read_csv(DATA_SOURCE, encoding='utf-8-sig')

print('=' * 60)
print('📊 TỔNG QUAN DỮ LIỆU')
print('=' * 60)
print(f'  Số dòng    : {df.shape[0]:,}')
print(f'  Số cột     : {df.shape[1]}')
print(f'  Bộ nhớ     : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print('=' * 60)

NameError: name 'pd' is not defined

In [ ]:
# Xem 5 dòng đầu
print('\n📋 5 DÒNG ĐẦU:')
df.head()


📋 5 DÒNG ĐẦU:


,user_id,product_category,age,dominant_gender,dominant_location,total_view,total_click,total_cart,total_wishlist,total_interactions,click_through_rate,cart_rate,unique_brands,avg_price,avg_discount,avg_purchase_value,avg_rating,user_total_categories,category_share,repurchase
0,U0001,Outdoor,44,Nam,Đà Nẵng,34,18,12,4,78,0.5294,0.6667,9,657000,9.7900,592000,4.2000,5,0.3000,1
1,U0001,Gia dụng,44,Nam,Đà Nẵng,8,5,2,3,21,0.6250,0.4000,3,7655000,16.0800,6424000,4.3000,5,0.0808,0
2,U0001,Điện tử,44,Nam,Đà Nẵng,12,9,3,12,47,0.7500,0.3333,9,9565000,10.1500,8594000,4.9000,5,0.1808,1
3,U0001,Sách,44,Nam,Đà Nẵng,38,23,12,8,90,0.6053,0.5217,6,247000,17.6100,203000,3.7000,5,0.3462,1
4,U0001,Làm đẹp,44,Nam,Đà Nẵng,6,4,2,7,24,0.6667,0.5000,1,349000,24.8800,262000,4.4000,5,0.0923,1


In [ ]:
# Xem kiểu dữ liệu và missing values
print('📋 THÔNG TIN KIỂU DỮ LIỆU VÀ MISSING VALUES:')
info_df = pd.DataFrame({
    'Dtype'   : df.dtypes,
    'Non-Null': df.count(),
    'Null'    : df.isnull().sum(),
    'Null %'  : (df.isnull().sum() / len(df) * 100).round(2)
})
info_df

📋 THÔNG TIN KIỂU DỮ LIỆU VÀ MISSING VALUES:


,Dtype,Non-Null,Null,Null %
user_id,str,61728,0,0.0000
product_category,str,61728,0,0.0000
age,int64,61728,0,0.0000
dominant_gender,str,61728,0,0.0000
dominant_location,str,61728,0,0.0000
total_view,int64,61728,0,0.0000
total_click,int64,61728,0,0.0000
total_cart,int64,61728,0,0.0000
total_wishlist,int64,61728,0,0.0000
total_interactions,int64,61728,0,0.0000


In [ ]:
# Thống kê mô tả cho numeric columns
print('📊 THỐNG KÊ MÔ TẢ (numeric columns):')
df.describe().T

📊 THỐNG KÊ MÔ TẢ (numeric columns):


,count,mean,std,min,25%,50%,75%,max
age,"61,728.0000",32.8711,13.2510,18.0000,21.0000,29.0000,42.0000,64.0000
total_view,"61,728.0000",12.7963,11.0346,0.0000,5.0000,9.0000,17.0000,45.0000
total_click,"61,728.0000",7.1824,7.0428,0.0000,2.0000,5.0000,10.0000,38.0000
total_cart,"61,728.0000",4.0902,3.4089,0.0000,2.0000,3.0000,5.0000,25.0000
total_wishlist,"61,728.0000",5.8463,4.7961,0.0000,2.0000,5.0000,8.0000,20.0000
total_interactions,"61,728.0000",33.5461,26.0246,0.0000,14.0000,25.0000,44.0000,130.0000
click_through_rate,"61,728.0000",0.4910,0.2135,0.0000,0.3750,0.5000,0.6667,0.8462
cart_rate,"61,728.0000",0.6381,0.5454,0.0000,0.3333,0.5385,0.8000,3.0000
unique_brands,"61,728.0000",4.9375,3.2035,1.0000,3.0000,4.0000,7.0000,14.0000
avg_price,"61,728.0000","4,387,056.3278","8,998,329.0914","30,000.0000","404,000.0000","1,295,500.0000","4,359,000.0000","79,953,000.0000"


## 2. Khám phá Schema — 20 cột

In [ ]:
# ============================================================
# SECTION 2: Schema 20 cột — pred_repurchase_dataset.csv
# ============================================================
schema = {
    'user_id'              : ('string (key)', 'Identifier — KHÔNG dùng làm feature'),
    'product_category'     : ('string (key+cat)', 'Key + Categorical — cần encode nếu dùng'),
    'age'                  : ('int', 'Tuổi người dùng — Demographic feature'),
    'dominant_gender'      : ('string (cat)', 'Giới tính phổ biến nhất — cần encode'),
    'dominant_location'    : ('string (cat)', 'Địa điểm phổ biến nhất — cần encode'),
    'total_view'           : ('int ≥ 0', 'Tổng lần xem sản phẩm'),
    'total_click'          : ('int ≥ 0', 'Tổng lần click'),
    'total_cart'           : ('int ≥ 0', 'Số lần thêm giỏ hàng'),
    'total_wishlist'       : ('int ≥ 0', 'Số lần thêm vào wishlist'),
    'total_interactions'   : ('int ≥ 0', 'Tổng view+click+cart+wishlist+purchase'),
    'click_through_rate'   : ('float [0,1]', 'click/view — funnel quality'),
    'cart_rate'            : ('float ≥ 0', 'cart/click — CÓ THỂ > 1 (direct-to-cart, HỢP LỆ)'),
    'unique_brands'        : ('int ≥ 1', 'Số thương hiệu đã tương tác'),
    'avg_price'            : ('int (VNĐ)', 'Giá niêm yết trung bình'),
    'avg_discount'         : ('float [0,45]%', 'Chiết khấu trung bình'),
    'avg_purchase_value'   : ('int (VNĐ)', 'Giá trị thực tế sau chiết khấu (= 0 nếu chưa mua)'),
    'avg_rating'           : ('float [1,5] / NaN', '~28% NaN — user chưa đánh giá'),
    'user_total_categories': ('int [1,15]', 'Số category user có mặt'),
    'category_share'       : ('float [0,1]', 'Tỷ lệ interaction category này / tổng user'),
    'repurchase'           : ('int 0/1', '🎯 TARGET — 1=tái mua (purchase_count≥4), 0=không'),
}

schema_df = pd.DataFrame(schema, index=['Type', 'Description']).T
schema_df.index.name = 'Column'
schema_df


,Type,Description
Column,,
user_id,string (key),Identifier — KHÔNG dùng làm feature
product_category,string (key+cat),Key + Categorical — cần encode nếu dùng
age,int,Tuổi người dùng — Demographic feature
dominant_gender,string (cat),Giới tính phổ biến nhất — cần encode
dominant_location,string (cat),Địa điểm phổ biến nhất — cần encode
total_view,int ≥ 0,Tổng lần xem sản phẩm
total_click,int ≥ 0,Tổng lần click
total_cart,int ≥ 0,Số lần thêm giỏ hàng
total_wishlist,int ≥ 0,Số lần thêm vào wishlist


In [ ]:
# Kiểm tra unique values cho categorical columns
cat_cols = ['dominant_gender', 'dominant_location', 'product_category']
for col in cat_cols:
    if col in df.columns:
        print(f'  {col}: {df[col].nunique()} giá trị unique')
        print(f'    → {sorted(df[col].dropna().unique().tolist())[:10]}')
        print()

  dominant_gender: 2 giá trị unique
    → ['Nam', 'Nữ']

  dominant_location: 5 giá trị unique
    → ['Cần Thơ', 'Hà Nội', 'Hải Phòng', 'Hồ Chí Minh', 'Đà Nẵng']

  product_category: 15 giá trị unique
    → ['Decor', 'Fitness', 'Gaming', 'Gia dụng', 'Healthy food', 'Luxury', 'Làm đẹp', 'Mẹ & Bé', 'Outdoor', 'Pet care']



## 3. Giải thích nguồn gốc nhãn `repurchase`

### 🏷️ Cơ chế gán nhãn: Otsu Thresholding trực tiếp trên `purchase_count`

Nhãn `repurchase` được xây dựng **hoàn toàn tự động** từ biến `purchase_count` — số lần khách hàng đã mua trong cùng `product_category` — qua 2 bước chặt chẽ về mặt thống kê.

---

#### Tại sao chọn `purchase_count`?

`purchase_count` là tín hiệu **trực tiếp và mạnh nhất** về hành vi tái mua:
- Mỗi dòng = 1 cặp `(user_id, product_category)` duy nhất
- `purchase_count` = số lần thực sự đã mua trong category đó
- Khách mua nhiều lần → **định nghĩa nghiệp vụ** của hành vi tái mua (RFM Frequency)

---

#### Bước 1 — Otsu Thresholding tự động (Otsu, 1979)

Thuật toán **Otsu** tìm ngưỡng $T^*$ tối đa hoá **inter-class variance** giữa 2 nhóm — **không đặt ngưỡng tay**:

$$T^* = \arg\max_T \;\omega_0(T)\,\omega_1(T)\,[\mu_0(T)-\mu_1(T)]^2$$

| Ký hiệu | Ý nghĩa |
|---------|---------|
| $\omega_0, \omega_1$ | Tỷ lệ nhóm "không tái mua" và "tái mua" tại ngưỡng $T$ |
| $\mu_0, \mu_1$ | Trung bình `purchase_count` của 2 nhóm |
| Quét | 500 ngưỡng từ min đến max của `purchase_count` |

> **Ưu điểm vs Q2 cứng:** Q2 = median cố định không tính đến hình dạng phân phối thực tế. Otsu tối ưu hóa sự phân tách giữa 2 nhóm theo dữ liệu thực.

---

#### Bước 2 — Gán nhãn & Loại bỏ `purchase_count` (Anti-Leakage)

$$\text{repurchase} = \begin{cases} 1 & \text{nếu } \texttt{purchase\_count} \geq T^* = 4.02 \\ 0 & \text{nếu } \texttt{purchase\_count} < T^* = 4.02 \end{cases}$$

| Nhãn | Ý nghĩa | Số lượng | Tỷ lệ |
|------|---------|----------|-------|
| **1** — Tái mua | Mua ≥ 4.02 lần trong category | 20,221 | **32.8%** |
| **0** — Không tái mua | Mua < 4.02 lần trong category | 41,507 | **67.2%** |

> ⚠️ **Anti-Leakage:** Ngay sau khi gán nhãn, `purchase_count` bị **xóa hoàn toàn** khỏi dataset. Feature matrix không chứa thông tin trực tiếp về số lần mua — model phải học từ hành vi gián tiếp (click, view, cart, wishlist, v.v.).

---

#### Dataset xuất ra (`pred_repurchase_dataset.csv`)

| Hạng mục | Chi tiết |
|----------|---------|
| Số dòng | 61,728 |
| Số cột | 20 (19 features + 1 target `repurchase`) |
| `purchase_count` | ✅ Đã xóa khỏi features (dùng để tạo nhãn Otsu) |
| Otsu $T^*$ | **4.02** lần |
| Phân tích chi tiết | `stat_analysis_repurchase.ipynb` |

In [ ]:
# ============================================================
# SECTION 3: Phân tích sơ bộ target và key features
# ============================================================
print('🎯 PHÂN PHỐI TARGET (repurchase):')
target_counts = df['repurchase'].value_counts()
target_pct    = df['repurchase'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count'     : target_counts,
    'Percentage': target_pct.round(2)
})
target_summary.index = ['Không tái mua (0)', 'Tái mua (1)']
print(target_summary.to_string())

ratio = target_counts[1] / target_counts[0]
print(f'\n  Imbalance ratio (1:0) = {ratio:.3f}')

🎯 PHÂN PHỐI TARGET (repurchase):
                   Count  Percentage
Không tái mua (0)  41507     67.2400
Tái mua (1)        20221     32.7600

  Imbalance ratio (1:0) = 0.487


In [ ]:
# Kiểm tra các giá trị đặc biệt cần lưu ý
print('⚠️ KIỂM TRA GIÁ TRỊ ĐẶC BIỆT:')
print()

# cart_rate có thể > 1
cart_over1 = (df['cart_rate'] > 1).sum()
print(f'  cart_rate > 1  : {cart_over1:,} dòng ({cart_over1/len(df)*100:.1f}%)  ← HỢP LỆ (direct-to-cart)')
print(f'  cart_rate max  : {df["cart_rate"].max():.4f}')

# avg_rating NaN
rating_nan = df['avg_rating'].isnull().sum()
print(f'\n  avg_rating NaN : {rating_nan:,} dòng ({rating_nan/len(df)*100:.1f}%)  ← User chưa đánh giá')
print(f'  Strategy: impute bằng median (4.0) trong NB3 preprocessing')

# avg_purchase_value = 0 khi chưa mua
purchase0 = (df['avg_purchase_value'] == 0).sum()
print(f'\n  avg_purchase_value = 0: {purchase0:,} dòng ({purchase0/len(df)*100:.1f}%)  ← Chưa mua lần nào')

# purchase_count đã bị xóa — không còn trong df
print(f'\n  purchase_count  : ✅ Đã xóa khỏi dataset (dùng để tạo nhãn Otsu T*=4.02)')
print(f'                       → Không có trong pred_repurchase_dataset.csv')

# Target distribution
print(f'\n🎯 PHÂN PHỐI TARGET (repurchase):')
print(f'  Tái mua (1)       : {(df["repurchase"]==1).sum():,}  ({(df["repurchase"]==1).mean()*100:.1f}%)')
print(f'  Không tái mua (0) : {(df["repurchase"]==0).sum():,}  ({(df["repurchase"]==0).mean()*100:.1f}%)')

⚠️ KIỂM TRA GIÁ TRỊ ĐẶC BIỆT:

  cart_rate > 1  : 6,344 dòng (10.3%)  ← HỢP LỆ (direct-to-cart)
  cart_rate max  : 3.0000

  avg_rating NaN : 17,306 dòng (28.0%)  ← User chưa đánh giá
  Strategy: impute bằng median (4.0) trong NB3 preprocessing

  avg_purchase_value = 0: 8,732 dòng (14.1%)  ← Chưa mua lần nào

  purchase_count  : ✅ Đã xóa khỏi dataset (dùng để tạo nhãn Otsu T*=4.02)
                       → Không có trong pred_repurchase_dataset.csv

🎯 PHÂN PHỐI TARGET (repurchase):
  Tái mua (1)       : 20,221  (32.8%)
  Không tái mua (0) : 41,507  (67.2%)


## 4. Lưu bản copy sang `data/synthetic_data.csv`

In [ ]:
# ============================================================
# SECTION 4: Lưu bản copy
# ============================================================
SYNTHETIC_PATH = DATA_DIR / 'synthetic_data.csv'
df.to_csv(SYNTHETIC_PATH, index=False, encoding='utf-8-sig')

print(f'✅ Đã lưu: {SYNTHETIC_PATH}')
print(f'   Shape: {df.shape}')

# Verify
df_check = pd.read_csv(SYNTHETIC_PATH, encoding='utf-8-sig')
assert df_check.shape == df.shape, 'Shape không khớp!'
print(f'   Verified: shape khớp ✓')

✅ Đã lưu: C:\Users\ADMIN\MOCK PROJECT VTI\mini_project_ml\data\synthetic_data.csv
   Shape: (61728, 20)
   Verified: shape khớp ✓


## 5. Tóm tắt Notebook 1

| Hạng mục | Kết quả |
|----------|---------|
| Dataset | 61,728 dòng × 20 cột |
| Key | `(user_id, product_category)` |
| Target | `repurchase` (0/1) — Otsu $T^*$ = 4.02 lần |
| Class imbalance | **32.8%** tái mua, **67.2%** không tái mua |
| Missing values | `avg_rating` ~28% NaN → impute median 4.0 tại NB3 |
| Giá trị đặc biệt | `cart_rate` có thể > 1 (hợp lệ — direct-to-cart) |
| Anti-leakage | `purchase_count` ✅ đã xóa khỏi feature matrix |
| Primary metric | **F1-Score** |
| Output | `data/synthetic_data.csv` ✅ |

**→ Notebook 2:** EDA đầy đủ — phân phối, correlation, outlier, feature importance sơ bộ


# 📦 Notebook 1 — Data Generation & Understanding

**Mục tiêu notebook:**
- Load `pred_repurchase_dataset.csv` (đã xử lý anti-leakage) và kiểm tra schema
- Hiểu cấu trúc 20 cột và ý nghĩa business của từng cột
- Mô tả bài toán dự đoán tái mua hàng (binary classification)
- Giải thích nguồn gốc nhãn `repurchase` (Otsu thresholding T*=4.02 trên purchase_count)
- Lưu bản copy vào `data/synthetic_data.csv` để EDA ở Notebook 2

---

## 🗺️ Lộ trình 15 bước — Pipeline ML đầy đủ

| # | Bước | Notebook |
|:-:|------|----------|
| **1** | **Đặt vấn đề** | **📌 NB1** |
| **2** | **Thu thập dữ liệu** | **📌 NB1** |
| 3 | Tiền xử lý & EDA | NB2 |
| 4 | Định nghĩa Target | NB2 |
| 5 | Feature Engineering | NB3 |
| 6 | Feature Selection | NB3 |
| 7 | Train/Test Split + Scale | NB3 |
| 8 | Chọn mô hình | NB3 |
| 9 | Huấn luyện | NB3 |
| 10 | Đánh giá baseline | NB3 |
| 11 | Feature Reduction | NB3 |
| 12 | So sánh & Phân tích sâu | NB4 |
| 13 | Kết luận Model tốt nhất | NB4 |
| 14 | Threshold Optimization | NB4 |
| 15 | KPI Check | NB4 |

> 💡 **Notebook này bao phủ: Bước 1 & 2 / 15**

---
## 🎨 Visual Identity — Quy ước màu sắc toàn dự án

> Áp dụng nhất quán trên **tất cả 4 notebooks** để đảm bảo portfolio cohesion.

### Màu chính — Class colors

| Token | Hex | Dùng cho | Ví dụ |
|-------|-----|----------|-------|
| `c0` | `#F7CAD0` | Fill — Class 0 (Không tái mua) | Boxplot, histogram, bar |
| `c0_edge` | `#C97680` | Viền — Class 0 | Edge/outline |
| `c1` | `#A0C4FF` | Fill — Class 1 (Tái mua) | Boxplot, histogram, bar |
| `c1_edge` | `#4A80C4` | Viền — Class 1 | Edge/outline |

### Màu nền & layout

| Token | Hex | Dùng cho |
|-------|-----|----------|
| `bg` | `#FAFAFA` | Figure & axes background |
| `grid` | `#EAEAEA` | Gridlines (rất nhẹ) |
| `text` | `#3A3A3A` | Main text, axis labels, regular titles |
| `sub` | `#888888` | Caption, annotation, secondary info |

### Màu nhấn — Semantic highlight

| Token | Hex | Ý nghĩa |
|-------|-----|---------|
| `hi_title` | `#D96C6C` | **Title đỏ** — feature có class separation / predictive signal mạnh |
| `hi_bdr` | `#B0CCEE` | Border nhẹ — subplot chứa feature quan trọng |

### Quy tắc sử dụng

```
📌 RULE 1 — Class colors:
   Class 0 (không tái mua) → LUÔN dùng #F7CAD0 (pastel pink)
   Class 1 (tái mua)       → LUÔN dùng #A0C4FF (pastel sky blue)

📌 RULE 2 — Highlight title:
   Feature có class separation mạnh → title màu #D96C6C (bold)
   Feature thông thường             → title màu #3A3A3A (regular)
   KHÔNG dùng ⭐ hay ký hiệu đặc biệt trong title

📌 RULE 3 — Background:
   Figure facecolor = #FAFAFA   |   Grid color = #EAEAEA
   Không dùng white (#FFFFFF) hay transparent

📌 RULE 4 — Consistency:
   Mọi chart trong dự án đều dùng chung dict VI = {...}
   Khai báo một lần ở đầu cell, tham chiếu qua key
```

### Áp dụng theo notebook

| Notebook | Charts | Colors dùng |
|----------|--------|-------------|
| **NB1** (Data Generation) | Schema, label distribution | `c0`, `c1`, `bg`, `text` |
| **NB2** (EDA) | Boxplot grid, heatmap, distribution | All tokens + `hi_title`, `hi_bdr` |
| **NB3** (Model Building) | Feature importance, learning curves | `c0`, `c1`, `bg`, `text`, `sub` |
| **NB4** (Evaluation) | ROC, PR, Confusion Matrix, KPI | `c0`, `c1`, `bg`, `text`, `sub` |



---
## 📌 Bước 1 / 15 — Đặt vấn đề & Mô tả Bài toán Business

### ❓ Câu hỏi kinh doanh cốt lõi

> **"Dựa trên hành vi tương tác của khách hàng với một category sản phẩm,**  
> **khách có quay lại mua thêm sản phẩm trong chính category đó không?"**

Một nền tảng e-commerce Việt Nam muốn **cá nhân hóa chiến lược giữ chân khách hàng** theo từng danh mục sản phẩm. Thay vì chạy khuyến mãi đại trà, họ cần biết: *"Khách hàng X có khả năng mua lại trong danh mục Y không?"*

---

### 🏢 Bối cảnh & Business Impact

- **Vấn đề:** ~67% khách không quay lại category sau lần mua đầu → chi phí acquisition cao bị lãng phí
- **Giải pháp:** Dự đoán sớm khách nào có khả năng tái mua category → gửi voucher/reminder đúng người
- **ROI:** Gửi voucher đại trà tốn 30,000 VNĐ/người × toàn bộ khách; nếu target đúng → chỉ gửi ~33% → tiết kiệm ~67% chi phí CRM

| Dự đoán | Thực tế | Hành động | Hệ quả |
|---------|---------|-----------|--------|
| 1 (tái mua) | 1 ✅ | Gửi voucher loyalty → giữ chân | Hiệu quả cao |
| 1 (tái mua) | 0 ❌ | Voucher gửi nhầm | Chi phí marketing lãng phí |
| 0 (không tái mua) | 1 ❌ | Không gửi gì | Bỏ lỡ cơ hội upsell → mất doanh thu |

---

### 🔍 Hiểu đúng "Tái mua" trong bài toán này

| Khái niệm | Định nghĩa trong bài toán này |
|-----------|-------------------------------|
| **Tái mua (repurchase = 1)** | Khách đã mua ở một category → **quay lại mua thêm ≥ 5 lần** trong cùng category đó |
| **Không tái mua (repurchase = 0)** | Khách chỉ mua 1–4 lần rồi dừng, hoặc không quay lại category đó nữa |
| **Đơn vị phân tích** | Mỗi cặp **(user, category)** — không phải toàn bộ lịch sử mua hàng |
| **Tín hiệu dự đoán** | Các feature trong data (KHÔNG dùng purchase_count) |

> 🎯 **Tại sao cùng category?** Khách mua áo thun → có tái mua áo thun nữa không?  
> Đây là **category loyalty** — khác với cross-sell (mua thêm category khác).  
> Model này giúp team CRM biết nên **re-target** khách vào đúng category họ hay mua.

---

### 🤖 Định nghĩa bài toán ML

| Hạng mục | Chi tiết |
|----------|---------|
| **Task** | Binary Classification |
| **Input (X)** | 16 raw behavioral features + 15 derived features (xem Bước 5) |
| **Output (y)** | `repurchase`: 0 = không tái mua category, 1 = tái mua category |
| **Nhãn nguồn gốc** | Otsu thresholding trên `purchase_count` → **T\* = 4.02** → label=1 nếu ≥ 5 lần |
| **Đơn vị** | 1 dòng = 1 user × 1 category (aggregate của toàn bộ hành vi trong category đó) |

---

### 📊 KPI mục tiêu

Vì class imbalance (~33/67), **F1-Score** là metric chính — cân bằng Precision và Recall:
- **Precision** cao → ít lãng phí voucher (ít FP)
- **Recall** cao → ít bỏ lỡ khách tái mua tiềm năng (ít FN)

| Metric | Ngưỡng | Lý do |
| Metric | Vai trò ở giai đoạn NB1 | Cơ sở phân tích hiện có |
|--------|--------------------------|------------------------|
| F1-Score | **Primary metric** | Dataset có class imbalance khoảng **67/33**; nếu luôn dự đoán lớp 0 thì Accuracy vẫn cao nhưng mô hình không có giá trị thực tiễn |
| Recall | **Guardrail metric** | Cần theo dõi riêng nguy cơ bỏ sót khách có khả năng tái mua; mức chi phí FN/FP cụ thể sẽ được lượng hóa ở notebook đánh giá |
| ROC-AUC | **Ranking metric** | Đo khả năng phân biệt hai nhóm khách hàng một cách độc lập với threshold, phù hợp để so sánh các mô hình ở giai đoạn sau |

> 📌 **Lưu ý phương pháp:** Ở notebook đầu tiên, ta mới chỉ có **business brief + mô tả dữ liệu + phân phối target sơ bộ**, nên **chưa đủ cơ sở khoa học để chốt trước các ngưỡng KPI định lượng** như F1 ≥ x hay ROC-AUC ≥ y.
>
> Các **ngưỡng KPI số** sẽ được xác lập sau khi hoàn thành:
> 1. **EDA và kiểm tra chất lượng dữ liệu** (NB2)
> 2. **Baseline model và feature engineering** (NB3)
> 3. **Model evaluation + threshold tuning + business cost analysis** (NB4)
>
> Vì vậy, ở **NB1** phần này nên được hiểu là **tiêu chí đánh giá ban đầu** chứ không phải **KPI kết quả đã chốt sẵn**.

**3 Models:** Logistic Regression (baseline) → XGBoost → LightGBM

In [ ]:
# ============================================================
# SECTION 0: Import thư viện và thiết lập đường dẫn
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Thiết lập display ---
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_colwidth', 50)

# --- Đường dẫn ---
# pred_repurchase_dataset.csv: đã loại purchase_count, có nhãn repurchase (Otsu T*=4.02)
DATA_SOURCE = Path(r'C:\Users\ADMIN\MOCK PROJECT VTI\pred_repurchase_dataset.csv')
BASE_DIR    = Path('.').resolve()   # mini_project_ml/
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models'

# Tạo thư mục nếu chưa có
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print(f'✅ Base directory : {BASE_DIR}')
print(f'✅ Data directory : {DATA_DIR}')
print(f'✅ Models directory: {MODELS_DIR}')
print(f'✅ Data source    : {DATA_SOURCE}')
print(f'   Exists? {DATA_SOURCE.exists()}')
print()
print('   📌 Nguồn: pred_repurchase_dataset.csv')
print('   ├─ purchase_count đã bị XÓA (dùng để tạo nhãn — anti-leakage)')
print('   ├─ repurchase: Otsu T* = 4.02 (label=1 nếu purchase_count ≥ 4.02)')
print('   └─ 20 cột còn lại: tất cả an toàn để làm feature')


✅ Base directory : C:\Users\ADMIN\MOCK PROJECT VTI\mini_project_ml
✅ Data directory : C:\Users\ADMIN\MOCK PROJECT VTI\mini_project_ml\data
✅ Models directory: C:\Users\ADMIN\MOCK PROJECT VTI\mini_project_ml\models
✅ Data source    : C:\Users\ADMIN\MOCK PROJECT VTI\pred_repurchase_dataset.csv
   Exists? True

   📌 Nguồn: pred_repurchase_dataset.csv
   ├─ purchase_count đã bị XÓA (dùng để tạo nhãn — anti-leakage)
   ├─ repurchase: Otsu T* = 4.02 (label=1 nếu purchase_count ≥ 4.02)
   └─ 20 cột còn lại: tất cả an toàn để làm feature



---
## 📌 Bước 2 / 15 — Thu thập dữ liệu (Data Collection)

**Nguồn dữ liệu:** `pred_repurchase_dataset.csv` — E-commerce behavioral dataset

### Cấu trúc dữ liệu

> **Mỗi dòng = 1 user × 1 category** — aggregate toàn bộ hành vi của user trong category đó.

| Nhóm cột | Ví dụ | Vai trò |
|----------|-------|---------|
| **Behavioral** | `total_view`, `total_click`, `total_cart`, `total_wishlist` | Tín hiệu hành vi trong category |
| **Purchase aggregate** | `avg_purchase_value`, `avg_price`, `avg_discount` | Thống kê giá/mua trong category |
| **Category profile** | `category_share`, `user_total_categories`, `unique_brands` | Mức độ tập trung vào category |
| **User profile** | `age`, `gender`, `membership_tier`, `region` | Đặc điểm demographics |
| **Rating** | `avg_rating` | Hài lòng trung bình với sản phẩm trong category (~28% missing) |
| **Target** | `repurchase` | 0/1 — tái mua trong category không? |

**Quy mô:** 61,728 cặp (user, category) × 20 cột  
**Class distribution:** 67.2% không tái mua (label=0) · 32.8% tái mua (label=1)  
**Otsu T\* = 4.02** → `purchase_count ≥ 5` → label=1

**Anti-leakage đã xử lý:**
- `purchase_count` → **ĐÃ XÓA** (dùng để tạo nhãn — nếu giữ = biết đáp án trước)
- `total_interactions` → **ĐÃ XÓA** (= view + click + cart + wishlist + purchase_count → chứa purchase_count)

### 1. Load dữ liệu nguồn


In [ ]:
# ============================================================
# SECTION 1: Load dữ liệu
# ============================================================
df = pd.read_csv(DATA_SOURCE, encoding='utf-8-sig')

print('=' * 60)
print('📊 TỔNG QUAN DỮ LIỆU')
print('=' * 60)
print(f'  Số dòng    : {df.shape[0]:,}')
print(f'  Số cột     : {df.shape[1]}')
print(f'  Bộ nhớ     : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print('=' * 60)

📊 TỔNG QUAN DỮ LIỆU
  Số dòng    : 61,728
  Số cột     : 20
  Bộ nhớ     : 11.14 MB


In [ ]:
# Xem 5 dòng đầu
print('\n📋 5 DÒNG ĐẦU:')
df.head()


📋 5 DÒNG ĐẦU:


,user_id,product_category,age,dominant_gender,dominant_location,total_view,total_click,total_cart,total_wishlist,total_interactions,click_through_rate,cart_rate,unique_brands,avg_price,avg_discount,avg_purchase_value,avg_rating,user_total_categories,category_share,repurchase
0,U0001,Outdoor,44,Nam,Đà Nẵng,34,18,12,4,78,0.5294,0.6667,9,657000,9.7900,592000,4.2000,5,0.3000,1
1,U0001,Gia dụng,44,Nam,Đà Nẵng,8,5,2,3,21,0.6250,0.4000,3,7655000,16.0800,6424000,4.3000,5,0.0808,0
2,U0001,Điện tử,44,Nam,Đà Nẵng,12,9,3,12,47,0.7500,0.3333,9,9565000,10.1500,8594000,4.9000,5,0.1808,1
3,U0001,Sách,44,Nam,Đà Nẵng,38,23,12,8,90,0.6053,0.5217,6,247000,17.6100,203000,3.7000,5,0.3462,1
4,U0001,Làm đẹp,44,Nam,Đà Nẵng,6,4,2,7,24,0.6667,0.5000,1,349000,24.8800,262000,4.4000,5,0.0923,1


In [ ]:
# Xem kiểu dữ liệu và missing values
print('📋 THÔNG TIN KIỂU DỮ LIỆU VÀ MISSING VALUES:')
info_df = pd.DataFrame({
    'Dtype'   : df.dtypes,
    'Non-Null': df.count(),
    'Null'    : df.isnull().sum(),
    'Null %'  : (df.isnull().sum() / len(df) * 100).round(2)
})
info_df

📋 THÔNG TIN KIỂU DỮ LIỆU VÀ MISSING VALUES:


,Dtype,Non-Null,Null,Null %
user_id,str,61728,0,0.0000
product_category,str,61728,0,0.0000
age,int64,61728,0,0.0000
dominant_gender,str,61728,0,0.0000
dominant_location,str,61728,0,0.0000
total_view,int64,61728,0,0.0000
total_click,int64,61728,0,0.0000
total_cart,int64,61728,0,0.0000
total_wishlist,int64,61728,0,0.0000
total_interactions,int64,61728,0,0.0000


In [ ]:
# Thống kê mô tả cho numeric columns
print('📊 THỐNG KÊ MÔ TẢ (numeric columns):')
df.describe().T

📊 THỐNG KÊ MÔ TẢ (numeric columns):


,count,mean,std,min,25%,50%,75%,max
age,"61,728.0000",32.8711,13.2510,18.0000,21.0000,29.0000,42.0000,64.0000
total_view,"61,728.0000",12.7963,11.0346,0.0000,5.0000,9.0000,17.0000,45.0000
total_click,"61,728.0000",7.1824,7.0428,0.0000,2.0000,5.0000,10.0000,38.0000
total_cart,"61,728.0000",4.0902,3.4089,0.0000,2.0000,3.0000,5.0000,25.0000
total_wishlist,"61,728.0000",5.8463,4.7961,0.0000,2.0000,5.0000,8.0000,20.0000
total_interactions,"61,728.0000",33.5461,26.0246,0.0000,14.0000,25.0000,44.0000,130.0000
click_through_rate,"61,728.0000",0.4910,0.2135,0.0000,0.3750,0.5000,0.6667,0.8462
cart_rate,"61,728.0000",0.6381,0.5454,0.0000,0.3333,0.5385,0.8000,3.0000
unique_brands,"61,728.0000",4.9375,3.2035,1.0000,3.0000,4.0000,7.0000,14.0000
avg_price,"61,728.0000","4,387,056.3278","8,998,329.0914","30,000.0000","404,000.0000","1,295,500.0000","4,359,000.0000","79,953,000.0000"


## 2. Khám phá Schema — 20 cột

In [ ]:
# ============================================================
# SECTION 2: Schema 20 cột — pred_repurchase_dataset.csv
# purchase_count đã bị xóa (dùng để tạo nhãn repurchase — anti-leakage)
# ============================================================
schema = {
    'user_id'              : ('string (key)', 'Identifier — KHÔNG dùng làm feature'),
    'product_category'     : ('string (key+cat)', 'Key + Categorical — cần encode nếu dùng'),
    'age'                  : ('int', 'Tuổi người dùng — Demographic feature'),
    'dominant_gender'      : ('string (cat)', 'Giới tính phổ biến nhất — cần encode'),
    'dominant_location'    : ('string (cat)', 'Địa điểm phổ biến nhất — cần encode'),
    'total_view'           : ('int ≥ 0', 'Tổng lần xem sản phẩm'),
    'total_click'          : ('int ≥ 0', 'Tổng lần click'),
    'total_cart'           : ('int ≥ 0', 'Số lần thêm giỏ hàng'),
    'total_wishlist'       : ('int ≥ 0', 'Số lần thêm vào wishlist'),
    'total_interactions'   : ('int ≥ 0', 'Tổng view+click+cart+wishlist+purchase'),
    'click_through_rate'   : ('float [0,1]', 'click/view — funnel quality'),
    'cart_rate'            : ('float ≥ 0', 'cart/click — CÓ THỂ > 1 (direct-to-cart, HỢP LỆ)'),
    'unique_brands'        : ('int ≥ 1', 'Số thương hiệu đã tương tác'),
    'avg_price'            : ('int (VNĐ)', 'Giá niêm yết trung bình'),
    'avg_discount'         : ('float [0,45]%', 'Chiết khấu trung bình'),
    'avg_purchase_value'   : ('int (VNĐ)', 'Giá trị thực tế sau chiết khấu (= 0 nếu chưa mua)'),
    'avg_rating'           : ('float [1,5] / NaN', '~28% NaN — user chưa đánh giá'),
    'user_total_categories': ('int [1,15]', 'Số category user có mặt'),
    'category_share'       : ('float [0,1]', 'Tỷ lệ interaction category này / tổng user'),
    'repurchase'           : ('int 0/1', '🎯 TARGET — 1=tái mua (purchase_count≥4), 0=không'),
}

schema_df = pd.DataFrame(schema, index=['Type', 'Description']).T
schema_df.index.name = 'Column'
print('📋 SCHEMA 20 CỘT (purchase_count đã xóa — anti-leakage):')
schema_df


📋 SCHEMA 20 CỘT (purchase_count đã xóa — anti-leakage):


,Type,Description
Column,,
user_id,string (key),Identifier — KHÔNG dùng làm feature
product_category,string (key+cat),Key + Categorical — cần encode nếu dùng
age,int,Tuổi người dùng — Demographic feature
dominant_gender,string (cat),Giới tính phổ biến nhất — cần encode
dominant_location,string (cat),Địa điểm phổ biến nhất — cần encode
total_view,int ≥ 0,Tổng lần xem sản phẩm
total_click,int ≥ 0,Tổng lần click
total_cart,int ≥ 0,Số lần thêm giỏ hàng
total_wishlist,int ≥ 0,Số lần thêm vào wishlist


In [ ]:
# Kiểm tra unique values cho categorical columns
cat_cols = ['dominant_gender', 'dominant_location', 'product_category']
for col in cat_cols:
    if col in df.columns:
        print(f'  {col}: {df[col].nunique()} giá trị unique')
        print(f'    → {sorted(df[col].dropna().unique().tolist())[:10]}')
        print()

  dominant_gender: 2 giá trị unique
    → ['Nam', 'Nữ']

  dominant_location: 5 giá trị unique
    → ['Cần Thơ', 'Hà Nội', 'Hải Phòng', 'Hồ Chí Minh', 'Đà Nẵng']

  product_category: 15 giá trị unique
    → ['Decor', 'Fitness', 'Gaming', 'Gia dụng', 'Healthy food', 'Luxury', 'Làm đẹp', 'Mẹ & Bé', 'Outdoor', 'Pet care']



## 3. Giải thích nguồn gốc nhãn `repurchase`

### 🏷️ Cơ chế gán nhãn: Otsu Thresholding trực tiếp trên `purchase_count`

Nhãn `repurchase` được xây dựng **hoàn toàn tự động** từ biến `purchase_count` — số lần khách hàng đã mua trong cùng `product_category` — qua 2 bước chặt chẽ về mặt thống kê.

---

#### Tại sao chọn `purchase_count`?

`purchase_count` là tín hiệu **trực tiếp và mạnh nhất** về hành vi tái mua:
- Mỗi dòng = 1 cặp `(user_id, product_category)` duy nhất
- `purchase_count` = số lần thực sự đã mua trong category đó
- Khách mua nhiều lần → **định nghĩa nghiệp vụ** của hành vi tái mua (RFM Frequency)

---

#### Bước 1 — Otsu Thresholding tự động (Otsu, 1979)

Thuật toán **Otsu** tìm ngưỡng $T^*$ tối đa hoá **inter-class variance** giữa 2 nhóm — **không đặt ngưỡng tay**:

$$T^* = \arg\max_T \;\omega_0(T)\,\omega_1(T)\,[\mu_0(T)-\mu_1(T)]^2$$

| Ký hiệu | Ý nghĩa |
|---------|---------|
| $\omega_0, \omega_1$ | Tỷ lệ nhóm "không tái mua" và "tái mua" tại ngưỡng $T$ |
| $\mu_0, \mu_1$ | Trung bình `purchase_count` của 2 nhóm |
| Quét | 500 ngưỡng từ min đến max của `purchase_count` |

> **Ưu điểm vs Q2 cứng:** Q2 = median cố định không tính đến hình dạng phân phối thực tế. Otsu tối ưu hóa sự phân tách giữa 2 nhóm theo dữ liệu thực.

---

#### Bước 2 — Gán nhãn & Loại bỏ `purchase_count` (Anti-Leakage)

$$\text{repurchase} = \begin{cases} 1 & \text{nếu } \texttt{purchase\_count} \geq T^* = 4.02 \\ 0 & \text{nếu } \texttt{purchase\_count} < T^* = 4.02 \end{cases}$$

| Nhãn | Ý nghĩa | Số lượng | Tỷ lệ |
|------|---------|----------|-------|
| **1** — Tái mua | Mua ≥ 4.02 lần trong category | 20,221 | **32.8%** |
| **0** — Không tái mua | Mua < 4.02 lần trong category | 41,507 | **67.2%** |

> ⚠️ **Anti-Leakage:** Ngay sau khi gán nhãn, `purchase_count` bị **xóa hoàn toàn** khỏi dataset. Feature matrix không chứa thông tin trực tiếp về số lần mua — model phải học từ hành vi gián tiếp (click, view, cart, wishlist, v.v.).

---

#### Dataset xuất ra (`pred_repurchase_dataset.csv`)

| Hạng mục | Chi tiết |
|----------|---------|
| Số dòng | 61,728 |
| Số cột | 20 (19 features + 1 target `repurchase`) |
| `purchase_count` | ✅ Đã xóa khỏi features (dùng để tạo nhãn Otsu) |
| Otsu $T^*$ | **4.02** lần |
| Phân tích chi tiết | `stat_analysis_repurchase.ipynb` |

In [ ]:
# ============================================================
# SECTION 3: Phân tích sơ bộ target và key features
# ============================================================
print('🎯 PHÂN PHỐI TARGET (repurchase):')
target_counts = df['repurchase'].value_counts()
target_pct    = df['repurchase'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count'     : target_counts,
    'Percentage': target_pct.round(2)
})
target_summary.index = ['Không tái mua (0)', 'Tái mua (1)']
print(target_summary.to_string())

ratio = target_counts[1] / target_counts[0]
print(f'\n  Imbalance ratio (1:0) = {ratio:.3f}')

🎯 PHÂN PHỐI TARGET (repurchase):
                   Count  Percentage
Không tái mua (0)  41507     67.2400
Tái mua (1)        20221     32.7600

  Imbalance ratio (1:0) = 0.487


In [ ]:
# Kiểm tra các giá trị đặc biệt cần lưu ý
print('⚠️ KIỂM TRA GIÁ TRỊ ĐẶC BIỆT:')
print()

# cart_rate có thể > 1
cart_over1 = (df['cart_rate'] > 1).sum()
print(f'  cart_rate > 1  : {cart_over1:,} dòng ({cart_over1/len(df)*100:.1f}%)  ← HỢP LỆ (direct-to-cart)')
print(f'  cart_rate max  : {df["cart_rate"].max():.4f}')

# avg_rating NaN
rating_nan = df['avg_rating'].isnull().sum()
print(f'\n  avg_rating NaN : {rating_nan:,} dòng ({rating_nan/len(df)*100:.1f}%)  ← User chưa đánh giá')
print(f'  Strategy: impute bằng median (4.0) trong NB3 preprocessing')

# avg_purchase_value = 0 khi chưa mua
purchase0 = (df['avg_purchase_value'] == 0).sum()
print(f'\n  avg_purchase_value = 0: {purchase0:,} dòng ({purchase0/len(df)*100:.1f}%)  ← Chưa mua lần nào')

# purchase_count đã bị xóa — không còn trong df
print(f'\n  purchase_count  : ✅ Đã xóa khỏi dataset (dùng để tạo nhãn Otsu T*=4.02)')
print(f'                       → Không có trong pred_repurchase_dataset.csv')

# Target distribution
print(f'\n🎯 PHÂN PHỐI TARGET (repurchase):')
print(f'  Tái mua (1)       : {(df["repurchase"]==1).sum():,}  ({(df["repurchase"]==1).mean()*100:.1f}%)')
print(f'  Không tái mua (0) : {(df["repurchase"]==0).sum():,}  ({(df["repurchase"]==0).mean()*100:.1f}%)')

⚠️ KIỂM TRA GIÁ TRỊ ĐẶC BIỆT:

  cart_rate > 1  : 6,344 dòng (10.3%)  ← HỢP LỆ (direct-to-cart)
  cart_rate max  : 3.0000

  avg_rating NaN : 17,306 dòng (28.0%)  ← User chưa đánh giá
  Strategy: impute bằng median (4.0) trong NB3 preprocessing

  avg_purchase_value = 0: 8,732 dòng (14.1%)  ← Chưa mua lần nào

  purchase_count  : ✅ Đã xóa khỏi dataset (dùng để tạo nhãn Otsu T*=4.02)
                       → Không có trong pred_repurchase_dataset.csv

🎯 PHÂN PHỐI TARGET (repurchase):
  Tái mua (1)       : 20,221  (32.8%)
  Không tái mua (0) : 41,507  (67.2%)


## 4. Lưu bản copy sang `data/synthetic_data.csv`

In [ ]:
# ============================================================
# SECTION 4: Lưu bản copy
# ============================================================
SYNTHETIC_PATH = DATA_DIR / 'synthetic_data.csv'
df.to_csv(SYNTHETIC_PATH, index=False, encoding='utf-8-sig')

print(f'✅ Đã lưu: {SYNTHETIC_PATH}')
print(f'   Shape: {df.shape}')

# Verify
df_check = pd.read_csv(SYNTHETIC_PATH, encoding='utf-8-sig')
assert df_check.shape == df.shape, 'Shape không khớp!'
print(f'   Verified: shape khớp ✓')

✅ Đã lưu: C:\Users\ADMIN\MOCK PROJECT VTI\mini_project_ml\data\synthetic_data.csv
   Shape: (61728, 20)
   Verified: shape khớp ✓


## 5. Tóm tắt Notebook 1

| Hạng mục | Kết quả |
|----------|---------|
| Dataset | 61,728 dòng × 20 cột |
| Key | `(user_id, product_category)` |
| Target | `repurchase` (0/1) — Otsu $T^*$ = 4.02 lần |
| Class imbalance | **32.8%** tái mua, **67.2%** không tái mua |
| Missing values | `avg_rating` ~28% NaN → impute median 4.0 tại NB3 |
| Giá trị đặc biệt | `cart_rate` có thể > 1 (hợp lệ — direct-to-cart) |
| Anti-leakage | `purchase_count` ✅ đã xóa khỏi feature matrix |
| Primary metric | **F1-Score** |
| Output | `data/synthetic_data.csv` ✅ |

**→ Notebook 2:** EDA đầy đủ — phân phối, correlation, outlier, feature importance sơ bộ